# BPSO against GA — analysis for Section 5.1

Both re-optimized with the corrected pipeline (compiled simulator, feasibility before the cache, independent-seed final evaluation), on the same 128 instances, seeds 1000 + k. Budgets: about 2,700 evaluations requested by the PSO (100 particles, 27 rounds) and about 2,400 by the GA (100 individuals, 50 generations). The GA's initial population contains full cross-training; the PSO's swarm does not.

In [1]:
import os, json, numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import rerun_analysis as ra
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
plt.rcParams.update({"font.family": "serif", "mathtext.fontset": "cm", "font.size": 8})
BASE = os.environ.get("ANALYSIS_BASE", "..")            # folder that holds the three result folders
OLD, PSO, GA = (os.path.join(BASE, f) for f in ("largeCases", "PSO_results_rerun", "GA_results_rerun"))
os.makedirs("figs", exist_ok=True); os.makedirs("tables", exist_ok=True)

pso, ga = ra.load_folder(PSO, "PSO"), ra.load_folder(GA, "GA")
pair = pso.merge(ga, on=["case"] + list(ra.FACTORS), suffixes=("_pso", "_ga"))
print(f"PSO: {len(pso)} | GA: {len(ga)} | paired: {len(pair)}")
print("infeasible final designs: PSO", pso.loc[~pso.feasible, "case"].tolist() or "none", "| GA", ga.loc[~ga.feasible, "case"].tolist() or "none")

PSO: 128 | GA: 128 | paired: 128
infeasible final designs: PSO none | GA none


## 1. Cost of the final designs (independent seeds)

In [2]:
from scipy.stats import wilcoxon, ttest_rel
pair["gap"] = pair.total_ga / pair.total_pso - 1                 # positive = GA design costs more
TIE = 0.005                                                       # |gap| below 0.5% counted as a tie
print(f"GA vs PSO: mean {pair.gap.mean():+.2%}, median {pair.gap.median():+.2%}, quartiles {pair.gap.quantile(.25):+.2%} / {pair.gap.quantile(.75):+.2%}")
print(f"PSO better in {(pair.gap > TIE).sum()}, GA better in {(pair.gap < -TIE).sum()}, ties (|gap| < {TIE:.1%}) {(pair.gap.abs() <= TIE).sum()} of {len(pair)}")
if len(pair) > 5:
    print(f"Wilcoxon signed-rank on the cost ratios: p = {wilcoxon(np.log(pair.total_ga / pair.total_pso)).pvalue:.3g} | paired t on log costs: p = {ttest_rel(np.log(pair.total_ga), np.log(pair.total_pso)).pvalue:.3g}")
tab = ra.factor_summary(pair, "gap")
tab["level"] = [ra.H_VARIANT_LABELS.get(l, l) if f == "h_variant" else l for f, l in zip(tab.factor, tab.level)]
tab["factor"] = tab.factor.map(lambda f: ra.FACTORS.get(f, f))
tab.to_csv("tables/ga_vs_pso_by_factor.csv", index=False)
ra.latex_table(tab, "tables/ga_vs_pso_by_factor.tex", "Relative cost of the GA's design against the BPSO's (positive: the GA's design costs more), by test-bed factor.",
               "tab:ga_bpso_factor", {"mean": "{:+.2%}", "median": "{:+.2%}", "min": "{:+.2%}", "max": "{:+.2%}", "cases": "{:d}"})
tab.style.format({"mean": "{:+.2%}", "median": "{:+.2%}", "min": "{:+.2%}", "max": "{:+.2%}"})

GA vs PSO: mean -0.50%, median -0.66%, quartiles -2.51% / +0.48%
PSO better in 30, GA better in 69, ties (|gap| < 0.5%) 29 of 128
Wilcoxon signed-rank on the cost ratios: p = 0.00115 | paired t on log costs: p = 0.134


,factor,level,cases,mean,median,min,max
0,SKUs $N$,10.000000,64,+0.55%,-0.53%,-8.09%,+19.15%
1,SKUs $N$,20.000000,64,-1.55%,-0.67%,-13.44%,+9.95%
2,Technicians $M$,5.000000,64,-0.16%,-0.67%,-13.44%,+19.15%
3,Technicians $M$,10.000000,64,-0.84%,-0.64%,-9.38%,+9.66%
4,Utilization $\rho$,0.650000,64,-0.13%,-0.38%,-9.38%,+19.15%
5,Utilization $\rho$,0.800000,64,-0.86%,-0.67%,-13.44%,+9.95%
6,$h_{\min}$,1.000000,64,-0.10%,-0.58%,-9.38%,+19.15%
7,$h_{\min}$,100.000000,64,-0.89%,-0.67%,-13.44%,+18.21%
8,Holding cost/workload,IND,64,-0.66%,-0.67%,-8.09%,+19.15%
9,Holding cost/workload,HPB,64,-0.34%,-0.64%,-13.44%,+18.43%


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(6.3, 2.6))
lo, hi = pair[["total_pso", "total_ga"]].min().min(), pair[["total_pso", "total_ga"]].max().max()
axes[0].loglog(pair.total_pso, pair.total_ga, "o", ms=3, mfc="none", mec="black"); axes[0].plot([lo, hi], [lo, hi], color="0.5", lw=0.8)
axes[0].set_xlabel("BPSO design cost"); axes[0].set_ylabel("GA design cost")
axes[1].hist(100 * pair.gap, bins=30, color="0.6", edgecolor="black"); axes[1].axvline(0, color="black", lw=0.8)
axes[1].set_xlabel("GA cost above BPSO (%)"); axes[1].set_ylabel("cases")
fig.tight_layout(); fig.savefig("figs/ga_vs_pso_costs.pdf"); plt.show()

## 2. Savings against full cross-training, both algorithms

In [4]:
comb = pd.concat([pso.assign(alg="BPSO"), ga.assign(alg="GA")])
print(comb.groupby("alg").saving.describe(percentiles=[.25, .5, .75]).round(3))
facs = [f for f in ra.FACTORS if comb[f].nunique() > 1]
fig, axes = plt.subplots(1, max(len(facs), 1), figsize=(1.7 * max(len(facs), 1) + 0.8, 2.5), sharey=True, squeeze=False)
ra.boxplots_by_factor(axes[0], comb, "saving", ("alg", ["BPSO", "GA"]), ["0.6", "white"], ["BPSO", "GA"], factors=facs)
axes[0, 0].set_ylabel("saving vs full cross-training (%)")
fig.legend([Patch(facecolor=c, edgecolor="black") for c in ["0.6", "white"]], ["BPSO", "GA"], loc="upper center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.88)); fig.savefig("figs/savings_bpso_ga_by_factor.pdf"); plt.show()

      count   mean    std    min    25%    50%    75%    max
alg                                                         
BPSO  128.0  0.287  0.159  0.035  0.130  0.310  0.410  0.800
GA    128.0  0.288  0.170  0.034  0.119  0.308  0.428  0.812


## 3. Selection bias of each algorithm

In [5]:
for name, df in (("BPSO", pso), ("GA", ga)):
    b = df.search_total / df.total - 1
    print(f"{name}: search cost below the independent cost by {-b.mean():.2%} on average (median {-b.median():.2%}), in {(b < 0).mean():.0%} of cases")

BPSO: search cost below the independent cost by 0.26% on average (median 0.09%), in 70% of cases
GA: search cost below the independent cost by 0.31% on average (median 0.08%), in 77% of cases


## 4. Budget and run time

In [6]:
bt = pd.DataFrame({"evaluations requested (median)": [pso.evaluations.median(), ga.evaluations.median()],
                   "run time per case, hours (median)": [pso.runtime_h.median(), ga.runtime_h.median()],
                   "run time per case, hours (max)": [pso.runtime_h.max(), ga.runtime_h.max()]}, index=["BPSO", "GA"])
bt.to_csv("tables/budget_runtime.csv"); bt.round(2)

,evaluations requested (median),"run time per case, hours (median)","run time per case, hours (max)"
BPSO,2700.0,1.92,2.58
GA,2399.0,0.61,0.94


## 5. Convergence against evaluations

In [7]:
fig, ax = plt.subplots(figsize=(3.3, 2.4))
for name, df, ls in (("BPSO", pso, "-"), ("GA", ga, "--")):
    xs, ys = [], []
    for l in df.log:
        if not l: continue
        y = ra.normalized_curve(l)
        if y is None: continue
        ev = np.cumsum([e.get("nevals", 0) or 0 for e in l]); ok = np.isfinite(y)
        xs.append(ev[ok]); ys.append(y[ok])
    if not xs:
        print(f"{name}: no statistics in the logs"); continue
    grid = np.linspace(max(x[0] for x in xs), min(x[-1] for x in xs), 50)
    med = np.median([np.interp(grid, x, y) for x, y in zip(xs, ys)], axis=0)
    ax.plot(grid, 100 * (med - 1), ls, color="black", label=name)
ax.set_xlabel("evaluations requested"); ax.set_ylabel("best cost above own final (%)"); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig("figs/convergence_bpso_ga.pdf"); plt.show()

## 6. How different are the designs?

In [8]:
def same_design(a, b): return sorted(map(tuple, a)) == sorted(map(tuple, b))
pair["identical"] = [same_design(a, b) for a, b in zip(pair.X_pso, pair.X_ga)]
print(f"identical designs (up to the order of technicians): {pair.identical.sum()} of {len(pair)}")
cols = ["technicians", "skills", "skills per technician", "technicians per SKU"]
pd.DataFrame({"BPSO": pso[cols].mean(), "GA": ga[cols].mean()}).round(2)

identical designs (up to the order of technicians): 0 of 128


,BPSO,GA
technicians,7.10,7.20
skills,41.64,39.23
skills per technician,5.91,5.50
technicians per SKU,2.77,2.61


## 7. Per-case table

In [9]:
keep = ["case"] + list(ra.FACTORS) + ["total_pso", "total_ga", "gap", "total_ff_pso", "saving_pso", "saving_ga", "identical"]
pair[keep].to_csv("tables/ga_vs_pso_per_case.csv", index=False); print("tables/ga_vs_pso_per_case.csv:", len(pair), "rows")

tables/ga_vs_pso_per_case.csv: 128 rows
